### Aula 04 — Filtros, ordenação e limite

Esta aula conecta regras de negócio a expressões booleanas e compara os planos de `filter()`, `orderBy()` e `limit()`. O foco é distinguir seleção local de registros, ordenação global e busca dos maiores valores, usando os operadores e resultados efetivamente salvos.

#### 1. Sessão e funções SQL

As próximas células importam a API e as funções como `F`, e obtêm a sessão `aula_04_filtros_ordenacao`. As condições são expressões `Column` que o Spark pode analisar e otimizar antes de uma action solicitar os dados.


In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = (
    SparkSession.builder
    .appName("aula_04_filtros_ordenacao")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/18 00:11:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### 2. Leitura e contrato de tipos

O arquivo de clientes de `04_filtros_ordenacao` é lido com cabeçalho e inferência de schema. A inferência pode acessar os dados para descobrir os tipos; ela não significa que os filtros seguintes já foram avaliados ou armazenados.

A saída salva de `printSchema()` identifica dez campos. `idade` é inteiro, `renda_mensal` é `double`, `cidade` é string e `ativo` é booleano. Essa verificação orienta as comparações e a ordenação: ordenar renda numérica tem semântica diferente de ordenar texto. Todos os campos permitem nulos no schema, sem que isso comprove a presença de nulos nos registros.


In [3]:
df = spark.read.csv("/workspace/datasets/04_filtros_ordenacao/clientes.csv", header= True, inferSchema= True)

In [4]:
df.printSchema()

root
 |-- cliente_id: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- idade: integer (nullable = true)
 |-- cidade: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- profissao: string (nullable = true)
 |-- renda_mensal: double (nullable = true)
 |-- sexo: string (nullable = true)
 |-- ativo: boolean (nullable = true)
 |-- data_cadastro: date (nullable = true)



#### 3. Filtro simples e diferença entre representação e resultado

`df_maiores` mantém clientes com **idade maior ou igual a 35**, incluindo o valor de fronteira 35. A transformação é lazy e devolve outro DataFrame, preservando `df`. Esse filtro é **Narrow**: cada partição pode avaliar seus próprios registros sem redistribuí-los entre partições.

A expressão isolada `df_maiores` na célula seguinte apresenta apenas a representação do DataFrame e seus tipos no output salvo. Já `df_maiores.show()` solicita dados e mostra 20 linhas: o Cliente 32, com idade 35, aparece entre os resultados. Essa evidência confirma a inclusão da fronteira na amostra, mas não informa a contagem total dos elegíveis.


In [5]:
df_maiores = (
    df.filter(F.col("idade") >= 35)
)

In [6]:
df_maiores

DataFrame[cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date]

In [7]:
df_maiores.show()

+----------+----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|cliente_id|      nome|idade|          cidade|estado|    profissao|renda_mensal|sexo|ativo|data_cadastro|
+----------+----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|         4| Cliente 4|   64|        Salvador|    BA|     Analista|     4253.53|   F|false|   2023-01-03|
|         5| Cliente 5|   60|        Salvador|    BA|   Engenheiro|     2710.92|   M|false|   2023-09-15|
|         7| Cliente 7|   46|Feira de Santana|    BA|    Professor|      6175.2|   M| true|   2023-12-15|
|        10|Cliente 10|   56|Feira de Santana|    BA|   Engenheiro|     8888.41|   F| true|   2024-02-22|
|        11|Cliente 11|   68|       São Paulo|    SP|Desenvolvedor|     5135.92|   M|false|   2024-04-26|
|        13|Cliente 13|   63|Feira de Santana|    BA|Desenvolvedor|     12740.2|   M|false|   2023-06-05|
|        17|Cliente 17|   62|       São Paulo|

#### 4. Expressões booleanas e regra de negócio

##### 4.1. Alternativas com OR

`df_cidade` mantém registros de Salvador **ou** Recife. O operador `|` representa OR entre condições Spark; `&` representa AND e `~` representa negação. `and`, `or` e `not` do Python não são substitutos para combinar objetos `Column`.

Cada comparação deve ser delimitada por parênteses. Em regras que misturam AND e OR, os agrupamentos determinam a quais alternativas cada restrição se aplica. A célula apenas constrói `df_cidade`: não há exibição nem contagem salva dessa consulta para afirmar quais cidades ou quantas linhas foram retornadas.


In [8]:
df_cidade = (
    df.filter(
        ((F.col("cidade") == "Salvador") | (F.col("cidade") == "Recife"))
    )
)

##### 4.2. Três critérios alternativos de qualificação

`df_clientes_qualificados` aceita um cliente quando **idade ≥ 30 OU renda mensal ≥ 5.000 OU ativo**. Basta uma condição ser verdadeira; o código não exige os três critérios simultaneamente. Traduzir essa expressão para linguagem de negócio ajuda a revisar o significado antes de discutir custo.

Embora o schema de `ativo` seja booleano, o código original compara o campo à string `"true"`. A próxima saída permite observar como o analisador resolve essa diferença de tipos neste ambiente, sem alterar a implementação.


In [9]:
df_clientes_qualificados = (
    df.filter(
        ((F.col("idade") >= 30) | (F.col("renda_mensal") >= 5000) | (F.col("ativo") == "true")) 
    )
)

##### 4.3. Coerção de tipos e semântica de nulos no plano

No `Analyzed Logical Plan` salvo, `5000` recebe conversão para `double` e a string usada na comparação de `ativo` aparece como `cast(true as boolean)`. No plano otimizado, as constantes estão resolvidas e a comparação booleana foi simplificada para `ativo`.

O filtro composto por OR continua sobre `FileScan csv`. Ao contrário de uma conjunção em que cada comparação precisa ser satisfeita, não seria correto exigir que todos os três campos fossem não nulos: uma linha com idade nula ainda pode ser aceita quando a renda atende ao critério. Filtros SQL mantêm somente condições verdadeiras; falso e desconhecido (`null`) não passam. [Referência: semântica de nulos do Spark SQL](https://spark.apache.org/docs/3.5.8/sql-ref-null-semantics.html).

`PushedFilters` contém a expressão OR enviada ao leitor. Esse registro não comprova eliminação de arquivos CSV nem mede economia de leitura. Não existe `Exchange` no plano salvo desse filtro: a regra pode ser aplicada registro a registro dentro das partições.


In [10]:
df_clientes_qualificados.explain(True)

== Parsed Logical Plan ==
'Filter ((('idade >= 30) OR ('renda_mensal >= 5000)) OR ('ativo = true))
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
Filter (((idade#19 >= 30) OR (renda_mensal#23 >= cast(5000 as double))) OR (ativo#25 = cast(true as boolean)))
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
Filter (((idade#19 >= 30) OR (renda_mensal#23 >= 5000.0)) OR ativo#25)
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
*(1) Filter (((idade#19 >= 30) OR (renda_mensal#23 >= 5000.0)) OR ativo#25)
+

#### 5. Negação e valores desconhecidos

As próximas células constroem `df_inativos`, negando a comparação de `ativo` com `"true"`, e `df_fora_salvador`, negando a comparação da cidade com Salvador. No primeiro caso, o comportamento esperado após a coerção booleana é selecionar valores falsos; nulos não passam automaticamente a representar clientes inativos.

Para `df_fora_salvador`, a saída salva acrescenta `isnotnull(cidade)` no plano otimizado. Isso preserva a semântica: quando `cidade` é nula, a igualdade com Salvador é desconhecida, e sua negação também não é verdadeira. Assim, “fora de Salvador” nessa consulta não inclui cidades desconhecidas.

O plano físico observado contém `FileScan csv → Filter`, com `IsNotNull(cidade)` e `Not(EqualTo(cidade,Salvador))` em `PushedFilters`. `PartitionFilters: []` indica ausência de filtro de partições de armazenamento. Não há contagem ou tabela salva dos resultados dessas duas negações.


In [11]:
df_inativos = (
    df.filter(
        ~(F.col("ativo") == "true")
    )
)

In [12]:
df_fora_salvador = (
    df.filter(
        ~(F.col("cidade") == "Salvador")
    )
)

In [13]:
df_fora_salvador.explain(True)

== Parsed Logical Plan ==
'Filter NOT ('cidade = Salvador)
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
Filter NOT (cidade#20 = Salvador)
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
Filter (isnotnull(cidade#20) AND NOT (cidade#20 = Salvador))
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
*(1) Filter (isnotnull(cidade#20) AND NOT (cidade#20 = Salvador))
+- FileScan csv [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] Batche

#### 6. Ordenação global por renda

`df_ordenado` descreve uma ordenação decrescente de `renda_mensal`. Diferentemente de um filtro local, uma ordem global precisa coordenar os intervalos de valores entre partições quando a distribuição disponível não atende ao requisito. Isso pode exigir Shuffle; o plano seguinte mostra a estratégia escolhida para esta consulta.

A saída salva de `show()` apresenta os 20 maiores valores exibidos em ordem decrescente. As três primeiras rendas são 14998.33, 14974.64 e 14967.5. Não foi definida uma coluna de desempate, portanto não há garantia de ordem estável entre clientes com a mesma renda.

Essa prévia confirma valores apresentados pela action, mas não fornece métricas de duração ou de movimentação de dados. Além disso, uma action que pede somente uma prévia pode acrescentar um limite ao plano que executa; o `explain` do DataFrame ordenado deve ser interpretado como o plano dessa consulta, sem presumir que reproduz todo detalhe interno de `show()`.


In [14]:
df_ordenado = (
    df.orderBy(
        F.col("renda_mensal").desc()
    )
)

In [15]:
df_ordenado.show()

+----------+-----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|cliente_id|       nome|idade|          cidade|estado|    profissao|renda_mensal|sexo|ativo|data_cadastro|
+----------+-----------+-----+----------------+------+-------------+------------+----+-----+-------------+
|       212|Cliente 212|   19|Feira de Santana|    BA|Desenvolvedor|    14998.33|   M|false|   2025-01-21|
|        19| Cliente 19|   39|Feira de Santana|    BA|   Engenheiro|    14974.64|   M| true|   2024-08-12|
|       159|Cliente 159|   35|       São Paulo|    SP|Desenvolvedor|     14967.5|   M|false|   2024-02-16|
|        30| Cliente 30|   56|        Salvador|    BA|    Professor|     14830.7|   F|false|   2024-05-04|
|       255|Cliente 255|   49|Feira de Santana|    BA|Desenvolvedor|    14822.31|   F|false|   2023-10-05|
|       187|Cliente 187|   46|        Salvador|    BA|    Professor|    14726.64|   F|false|   2023-03-28|
|       229|Cliente 229|   27|       

##### 6.1. `Exchange rangepartitioning`, `Sort` e AQE

A saída salva de `df_ordenado.explain(True)` mostra, de baixo para cima:

| Operador | Papel nesta ordenação |
| --- | --- |
| `FileScan csv` | Produz os registros da fonte. |
| `Exchange rangepartitioning(..., 200)` | Planeja redistribuição por intervalos da renda para compatibilizar a ordem entre partições. Essa é uma fronteira planejada de Shuffle. |
| `Sort [... DESC NULLS LAST]` | Ordena os registros dentro da distribuição estabelecida; as rendas nulas ficam ao final. |
| `AdaptiveSparkPlan isFinalPlan=false` | Envolve o plano adaptativo; o texto salvo ainda não é um plano final de execução. |

As faixas do `Exchange` e a ordenação em cada partição se complementam para produzir a ordem global. Isso torna a operação **Wide** no plano apresentado. O número 200 é o alvo de partições desse Exchange planejado, não a contagem de executors nem prova de que exatamente 200 tasks concluíram. O Adaptive Query Execution pode ajustar aspectos da estratégia com estatísticas de execução. [Referência: AQE no Spark](https://spark.apache.org/docs/3.5.8/sql-performance-tuning.html#adaptive-query-execution).

Não há métricas do Spark UI ou plano final salvo para quantificar o Shuffle efetivamente executado. A evidência aqui é a escolha física impressa para a ordenação sem limite.


In [16]:
df_ordenado.explain(True)

== Parsed Logical Plan ==
'Sort ['renda_mensal DESC NULLS LAST], true
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
Sort [renda_mensal#23 DESC NULLS LAST], true
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
Sort [renda_mensal#23 DESC NULLS LAST], true
+- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [renda_mensal#23 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(renda_mensal#23 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=66]
     

#### 7. `limit(10)` sem ordenação

`df_limitado` limita a saída a até dez registros de `df`, sem definir quais têm prioridade. Não representa os dez clientes mais ricos, nem uma amostra aleatória. Sem uma ordem explícita, os registros escolhidos não devem ser tratados como uma seleção estável para uma regra de negócio.

O plano lógico salvo usa `LocalLimit 10` e `GlobalLimit 10`: limites por partição podem restringir candidatos, e o limite global controla o total de saída. O plano físico desta consulta usa `CollectLimit 10` sobre `FileScan csv`, sem `Sort`.

`limit()` é uma transformação. A inspeção do plano não apresenta as dez linhas nem prova que só dez registros seriam lidos da fonte: a quantidade de trabalho depende da execução e das operações anteriores. Também não é correto concluir que um limite simples sempre precisa percorrer todas as linhas. Neste bloco há apenas o plano salvo.


In [17]:
df_limitado = (
    df.limit(10)
)

In [18]:
df_limitado.explain(True)

== Parsed Logical Plan ==
GlobalLimit 10
+- LocalLimit 10
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
GlobalLimit 10
+- LocalLimit 10
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
GlobalLimit 10
+- LocalLimit 10
   +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
CollectLimit 10
+- FileScan csv [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 pat

#### 8. Ordenação com limite: a consulta Top-10

`df_top10` combina renda decrescente e `limit(10)`. Sua semântica é obter até dez linhas de maior renda, considerando a ordenação; é diferente do limite sem critério da seção anterior.

O Spark pode planejar a seleção de poucos candidatos em cada partição e depois combinar esses candidatos para encontrar o Top-10 global. Essa estratégia reduz o conjunto a ordenar e combinar integralmente, mas ainda precisa examinar os registros relevantes da fonte para identificar os maiores. Não implica ler apenas dez linhas do CSV. Empates de renda continuam sem critério adicional de desempate.


In [19]:
df_top10 = (
    df.orderBy(
        F.col("renda_mensal").desc()
    ).limit(10)
)

##### 8.1. `TakeOrderedAndProject` observado

O plano lógico ainda contém `Sort`, `LocalLimit` e `GlobalLimit`, mas o físico salvo usa **`TakeOrderedAndProject(limit=10, orderBy=[renda_mensal DESC NULLS LAST])`** sobre o scan. Não aparecem um `Sort` completo e um `Exchange rangepartitioning` separados, como no plano de `df_ordenado`.

O operador reúne seleção dos maiores e projeção da saída. Ele pode reduzir candidatos localmente antes da combinação global. A ausência de um `Exchange` explícito nesse texto não garante ausência universal de comunicação ou Shuffle interno quando houver múltiplas partições; o que esta saída demonstra é a estratégia de Top-K, em vez da ordenação global completa do bloco anterior. [Referência: implementação do operador no Spark](https://github.com/apache/spark/blob/v3.5.8/sql/core/src/main/scala/org/apache/spark/sql/execution/limit.scala#L247-L345).

Não há `show()` de `df_top10` nem medidas de execução salvas neste notebook. Portanto, não se atribui a este bloco uma tabela observada ou um ganho de tempo. Em Engenharia de Dados, comparar os operadores antes de estimar custo evita equiparar toda consulta com `orderBy()` à mesma estratégia física.


In [20]:
df_top10.explain(True)

== Parsed Logical Plan ==
GlobalLimit 10
+- LocalLimit 10
   +- Sort [renda_mensal#23 DESC NULLS LAST], true
      +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Analyzed Logical Plan ==
cliente_id: int, nome: string, idade: int, cidade: string, estado: string, profissao: string, renda_mensal: double, sexo: string, ativo: boolean, data_cadastro: date
GlobalLimit 10
+- LocalLimit 10
   +- Sort [renda_mensal#23 DESC NULLS LAST], true
      +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Optimized Logical Plan ==
GlobalLimit 10
+- LocalLimit 10
   +- Sort [renda_mensal#23 DESC NULLS LAST], true
      +- Relation [cliente_id#17,nome#18,idade#19,cidade#20,estado#21,profissao#22,renda_mensal#23,sexo#24,ativo#25,data_cadastro#26] csv

== Physical Plan ==
TakeOrderedAndProject(limit=10, orderBy=[renda_mensal#23 DESC NUL

#### 9. Encerramento

`spark.stop()` encerra a sessão após os experimentos. Nenhuma escrita de dados é realizada pelos blocos desta aula.


In [22]:
spark.stop()

#### Resumo dos aprendizados

- Filtros expressam regras de negócio: operadores, fronteiras e parênteses determinam os registros elegíveis.
- `filter()` mantém apenas condições verdadeiras; negar uma comparação com nulo não a torna verdadeira.
- Representar um DataFrame ou inspecionar seu plano não equivale a exibir suas linhas.
- A ordenação sem limite mostrou `Exchange rangepartitioning` e `Sort` em um plano adaptativo ainda não final.
- `limit(10)` sem ordem mostrou `CollectLimit`, sem definir os maiores valores.
- `orderBy(...).limit(10)` mostrou `TakeOrderedAndProject`, evidenciando uma estratégia de Top-K.
- Operadores planejados ajudam a prever custos; métricas de execução seriam necessárias para confirmar duração e movimentação efetivas.

Para aprofundar: [documentação da Aula 04](../docs/04_filtros_ordenacao.md).
